# Module 1: Local HuBMAP lung data verification and QC

**Biological question.** Can we verify the HuBMAP lung snRNA-seq teaching set on disk and prepare nuclei from each donor for cross-donor analysis?

**Learning objectives.**
- Map required course files to local paths and verify presence/size against a provenance record (Remember, Understand)
- Run single-nucleus QC and preprocessing with justified thresholds (Apply)

**Bloom.** Remember, Understand, Apply | **Instructional time.** 30 min | **Compute.** minutes per donor

**Prerequisites.** Basic Python / AnnData. CFDE Workbench vs DCC portals (screencast).

**Data (required course total).** Measured in `outputs/tables/module1_learner_download_summary.tsv`. HuBMAP snRNA `raw_expr.h5ad` + metadata for Donor_1 to Donor_4; Donor_2 `multiome_mofa.hdf5` (Module 3); GTEx TPM, reads, and annotations plus GEO and OSDR files (Module 4). Cite primary HuBMAP DOIs; download public processed releases.

**Cohort.** Donor_1 (37), Donor_2 (25), Donor_3 (56.8), Donor_4 (52.96).

**Run order.** Run the analysis sections in order; the notebook carries state between cells.


## How this module fits the course

This is where the course acquires its data and decides what is fit to analyse. Everything Modules 2
to 4 report rests on the cohort assembled here and on the thresholds justified here, so a QC choice
made in this notebook propagates to every later result.

The notebook works through five stages: locate the datasets through CFDE surfaces and the DCC
portal, verify what arrived against its provenance record, inventory what is on disk, apply and
justify QC on one teaching block, then repeat the QC across all four donors so Module 2 has its
inputs.

Read the download plan before running anything. Module 1 verifies every file the whole course needs,
including the GTEx, GEO and OSDR files that only Module 4 opens, and stops if any is missing.


## 0. Setup


In [ ]:
from pathlib import Path
import os
import sys

MODULE_ROOT = Path.cwd().resolve()
if MODULE_ROOT.name == "notebooks":
    MODULE_ROOT = MODULE_ROOT.parent
if not (MODULE_ROOT / "scripts").is_dir():
    MODULE_ROOT = next(
        (p for p in MODULE_ROOT.parents if (p / "scripts").is_dir()), MODULE_ROOT
    )
if not (MODULE_ROOT / "scripts").is_dir():
    raise FileNotFoundError(
        f"Cannot find scripts/ from cwd={Path.cwd()}. "
        "Open the notebook from the project root or from its notebooks/ folder."
    )
# Prefer project root as cwd so relative config/data paths resolve.
os.chdir(MODULE_ROOT)
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

from scripts.common.paths import load_config, ensure_output_dirs, resolve
from scripts.common.plotting import apply_figure_style
from scripts.common.io import environment_versions

cfg = load_config()
ensure_output_dirs(cfg)
apply_figure_style()
print("MODULE_ROOT:", MODULE_ROOT)
print("cohort:", cfg["cohort"]["donors"])
print("teaching inputs:", cfg["module2"]["inputs"])
print("live_query:", cfg["module1"]["discovery"]["live_query"])
print(environment_versions())


## Discovery: CFDE surfaces and the DCC portals

CFDE (Workbench, Data Matrix, registered APIs) helps discover Common Fund resources. **HuBMAP and GTEx portals remain authoritative for the analysis-ready files.** Obtain files via the DCC web UI or a programmatic download once URLs/asset IDs are known. APIs registered in SmartAPI (including HuBMAP's) can assist programmatic discovery; this notebook starts from files already on disk.



## 1. Identify and place teaching files

The table below is the **curated teaching cohort**, stamped locally for this course.



In [ ]:
import pandas as pd
qpath = resolve(cfg, "outputs_tables") / "module1_hubmap_api_query.tsv"
display(pd.read_csv(qpath, sep="\t"))
print("Discover is not cohort lock: reasons are in selection_reason.")


## 2. Inventory and measured-size verification


In [ ]:
from scripts.nb01_discovery.inventory import (
    inventory_all_donors, summarize_inventory, snrna_block_metadata_all, write_inventory_outputs
)
inventory = inventory_all_donors(cfg)
summary = summarize_inventory(inventory)
blocks = snrna_block_metadata_all(cfg)
display(blocks[blocks.get("is_teaching_block", True) == True] if "is_teaching_block" in blocks.columns else blocks)
inv_paths = write_inventory_outputs(cfg)
print(inv_paths)


### Verify required teaching files (measured sizes)

Missing required files are reported by name. Optional `expr` / `scvelo` / full MuData are listed but not required.


In [ ]:
from scripts.nb01_discovery.verify import write_verification_outputs, learner_download_manifest
ver_paths = write_verification_outputs(cfg)
manifest = learner_download_manifest(cfg)
display(manifest)
display(pd.read_csv(ver_paths["download_summary"], sep="\t"))
missing = manifest[(manifest["required"]) & (~manifest["exists"])]
assert missing.empty, f"Missing required files:\n{missing}"
print("All required teaching files present.")


## 3. Provenance and access table

Redistribution status is **not redistributed; learner downloads from source portal**.


In [ ]:
from scripts.nb01_discovery.access_table import build_access_provenance_table, write_access_provenance_table
access = build_access_provenance_table(cfg)
display(access.head(12))
print(access["redistribution_status"].unique().tolist())
access_path = write_access_provenance_table(cfg)
print("wrote", access_path)


## 4. Composition gate (airway-lineage epithelial labels)

Before locking the four-donor cohort for Module 2 DE, check whether airway-secretory-related Azimuth labels (Club, Goblet, Deuterosomal, SMG, Tuft) exist in the teaching blocks. Classic SMG labels may be absent; Club / transitional labels fold into the coarse `epithelial` class in Module 2 (four buckets: epithelial, endothelial, stromal, immune).


In [ ]:
from scripts.nb01_discovery.composition_gate import write_composition_gate
gate_paths = write_composition_gate(cfg)
display(pd.read_csv(gate_paths["airway_lineage"], sep="\t"))
display(pd.read_csv(gate_paths["airway_wide"], sep="\t"))


## 5. QC filters and preprocessing (one teaching block)

Thresholds live in `config/paths.yaml` -> `module1.qc` / `module1.preprocess`. Running QC across donors writes **per-block** logs (`module1_qc_filter_log_<block_id>.tsv`) plus a combined `module1_qc_filter_log_all_donors.tsv` so multi-donor runs do not overwrite each other.


In [ ]:
from scripts.nb02_qc.load import load_module2_adata
from scripts.nb02_qc.qc import apply_filters, compute_qc_metrics, qc_metrics_summary
from scripts.nb02_qc.preprocess import (
    append_subsample_filter_row,
    hvg_pca,
    maybe_subsample,
    neighbors_umap,
    normalize_log,
)
from scripts.nb02_qc.export import save_module1_outputs
from scripts.common.plotting import qc_violin_panel, qc_scatter, umap_panel
from scripts.common.runtime import finalize_timing, peak_rss_mb
import time

block_id = cfg["module1"]["block_id"]
donor_label = cfg["module1"]["donor_label"]
_m1_timing = {
    "_t0": time.perf_counter(),
    "_compute_seconds": None,
    "peak_rss_mb": None,
    "peak_rss_mb_start": peak_rss_mb(),
    "peak_rss_note": (
        "process peak RSS so far (ru_maxrss); monotone across a loop, not per-step"
    ),
}
adata, paths, join_info = load_module2_adata(cfg, block_id=block_id, donor_label=donor_label)
adata.uns["hubmap_donor_label"] = donor_label
print(paths["raw_expr"], adata.shape, join_info)

adata = compute_qc_metrics(adata)
fig_dir = resolve(cfg, "outputs_figures")
fig_paths = {
    "qc_violins": qc_violin_panel(
        adata, keys=["total_counts", "n_genes_by_counts", "percent_mito"],
        path=fig_dir / f"module1_{block_id}_qc_violins.png",
    ),
    "counts_vs_genes": qc_scatter(
        adata, x="total_counts", y="n_genes_by_counts",
        path=fig_dir / f"module1_{block_id}_qc_counts_vs_genes.png",
    ),
    "counts_vs_mito": qc_scatter(
        adata, x="total_counts", y="percent_mito",
        path=fig_dir / f"module1_{block_id}_qc_counts_vs_mito.png",
    ),
}
adata, filter_log = apply_filters(adata, cfg)
display(filter_log)
adata = normalize_log(adata, cfg)
adata = hvg_pca(adata, cfg)
adata, sub_info = maybe_subsample(adata, cfg)
filter_log = append_subsample_filter_row(filter_log, sub_info, adata)
adata = neighbors_umap(adata, cfg)
colors = [c for c in ["total_counts", "percent_mito", "azimuth_label"] if c in adata.obs.columns]
fig_paths["umap"] = umap_panel(adata, colors=colors, path=fig_dir / f"module1_{block_id}_umap.png")
finalize_timing(_m1_timing)
out = save_module1_outputs(
    cfg, adata, filter_log=filter_log, metrics_summary=qc_metrics_summary(adata),
    figure_paths=fig_paths,
    run_extras={
        "label_join": join_info,
        "subsample": sub_info,
        **{k: v for k, v in _m1_timing.items() if not str(k).startswith("_t")},
    },
)
print(out, f"_compute_seconds={_m1_timing.get('_compute_seconds')}", f"peak_rss_mb={_m1_timing.get('peak_rss_mb')}")


### QC all teaching donors (default)

Process every Module 2 input block. Each run writes a per-block filter log and refreshes
`module1_qc_filter_log_all_donors.tsv`. Budget compute time for four donors (teaching subsample
caps three of them at `module1.max_nuclei`). The single-block scaffold above is a dry-run for
Donor_1 only.


In [ ]:
from scripts.run_modules_smoke import run_module2

# Default: QC all four teaching donors so Module 2 has its inputs.
# CLI equivalent: python scripts/run_modules_smoke.py --module 2
for item in cfg["module2"]["inputs"]:
    run_module2(
        block_id=item.get("block_id") or item.get("primary_id"),
        donor_label=item["donor_label"],
    )


## Questions this result raises

Use the Module 1 tables and figures already written under `outputs/`.

1. Which QC threshold removed the most nuclei, and would a neighboring cut change the donor composition story?
2. What does Azimuth join coverage tell you about unlabeled barcodes on Donor_1 versus Donor_4?
3. What claim about airway lineages is **not** supported by this cohort after QC?
4. If a colleague treated HubMAP processed IDs as interchangeable with primary DOIs, what would go wrong?
